# Kaggle

In [26]:
from pathlib import Path
import requests
import zipfile

In [27]:
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/arnavr10880/concrete-crack-images-for-classification"

In [28]:
download_dir = Path.cwd() / "Kaggle Mendeley check"
zip_path = download_dir / "concrete-crack-images-for-classification.zip"
extract_dir = download_dir / "concrete-crack-images-for-classification"

download_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)

In [29]:
# Download the file
with requests.get(KAGGLE_URL, stream=True) as r:
	r.raise_for_status()
	
	with open(zip_path, "wb") as f:
		for chunk in r.iter_content(chunk_size=8192):
			if chunk:
				f.write(chunk)

print(f"Downloaded to: {zip_path}")

# Extract the zip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
	zip_ref.extractall(extract_dir)

print(f"Extracted to: {extract_dir}")

Downloaded to: w:\DeepLearning_Crack\Appendix\Kaggle Mendeley check\concrete-crack-images-for-classification.zip
Extracted to: w:\DeepLearning_Crack\Appendix\Kaggle Mendeley check\concrete-crack-images-for-classification


# Verifying that the 2 data sources are equal

In [30]:
import os
import hashlib
import shutil

In [31]:
def file_hash(path: str | Path, chunk_size: int = 8192) -> str:
	h = hashlib.sha256()
	with open(path, "rb") as f:
		while chunk := f.read(chunk_size):
			h.update(chunk)
	return h.hexdigest()

def directory_snapshot(root: str | Path) -> dict[str, tuple[str, str | None]]:
	snapshot: dict[str, tuple[str, str | None]] = {}
	for dirpath, dirnames, filenames in os.walk(root):
		rel_dir = os.path.relpath(dirpath, root)

		for dirname in dirnames:
			rel_path = os.path.normpath(os.path.join(rel_dir, dirname))
			snapshot[rel_path] = ("dir", None)

		for filename in filenames:
			full_path = os.path.join(dirpath, filename)
			rel_path = os.path.normpath(os.path.join(rel_dir, filename))
			snapshot[rel_path] = ("file", file_hash(full_path))

	return snapshot

def directories_equal(dir1: str | Path, dir2: str | Path) -> bool:
	return directory_snapshot(dir1) == directory_snapshot(dir2)

In [32]:
dir_kaggle = r"Kaggle Mendeley check\concrete-crack-images-for-classification"
dir_mendeley = "./../Dataset Folder/Original/Concrete Crack Images for Classification"

os.makedirs(name=rf"{dir_kaggle}\Negative", exist_ok=True)
open(os.path.join(rf"{dir_kaggle}\Negative", ".gitkeep"), "w").close()

os.makedirs(name=rf"{dir_kaggle}\Positive", exist_ok=True)
open(os.path.join(rf"{dir_kaggle}\Positive", ".gitkeep"), "w").close()

In [33]:
if directories_equal(dir_kaggle, dir_mendeley):
	print("Directories are equal")
else:
	print("Directories are different")

Directories are equal


In [34]:
shutil.rmtree(path=download_dir)